In [1]:
import os
import json

# Project Root Setup
PROJECT_ROOT = os.getcwd()
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

PATHS = {
    "chunks": os.path.join(DATA_PROCESSED_DIR, "chunks.json"),
    "faiss_index": os.path.join(DATA_PROCESSED_DIR, "faiss_index.bin"),
    "bm25_index": os.path.join(DATA_PROCESSED_DIR, "bm25_index.pkl"),
    "embeddings": os.path.join(DATA_PROCESSED_DIR, "embeddings.npy")
}

# Verify Stage 2 Chunk Dataset
if os.path.exists(PATHS["chunks"]):
    with open(PATHS["chunks"], "r", encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"Loaded chunks.json successfully! Total chunks to index: {len(chunks)}")
else:
    print("Not Found")

Loaded chunks.json successfully! Total chunks to index: 2353


In [2]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Selection: Fast 384-dimensional lightweight encoder
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

def build_dense_faiss_index():
    print("=== BUILDING DENSE FAISS VECTOR INDEX ===")

    with open(PATHS["chunks"], "r", encoding="utf-8") as f:
        chunks = json.load(f)

    texts = [c["chunk_text"] for c in chunks]

    print(f"Loading embedding model ({EMBEDDING_MODEL_NAME})...")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    print("Encoding chunks into dense vector embeddings...")
    embeddings = model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

    # Save raw numpy array
    np.save(PATHS["embeddings"], embeddings)

    # Construct FAISS Index using Inner Product (Cosine Similarity on normalized vectors)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings.astype('float32'))

    # Save FAISS Index
    faiss.write_index(index, PATHS["faiss_index"])

    print("\n=== DENSE INDEX COMPLETE ===")
    print(f"Total Vectors Indexed: {index.ntotal}")
    print(f"Saved FAISS Index to: {PATHS['faiss_index']}")

build_dense_faiss_index()

=== BUILDING DENSE FAISS VECTOR INDEX ===
Loading embedding model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding chunks into dense vector embeddings...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]


=== DENSE INDEX COMPLETE ===
Total Vectors Indexed: 2353
Saved FAISS Index to: E:\Rasengan\College-Knowledge-Search\data\processed\faiss_index.bin


In [3]:
import pickle
from rank_bm25 import BM25Okapi

def build_sparse_bm25_index():
    print("=== BUILDING SPARSE BM25 KEYWORD INDEX ===")

    with open(PATHS["chunks"], "r", encoding="utf-8") as f:
        chunks = json.load(f)

    # Lowercase tokenization for BM25 keyword matching
    corpus = [c["chunk_text"].lower().split() for c in chunks]

    print("Building BM25 Inverted Index...")
    bm25 = BM25Okapi(corpus)

    # Save BM25 Index
    with open(PATHS["bm25_index"], "wb") as f:
        pickle.dump(bm25, f)

    print("\n=== SPARSE INDEX COMPLETE ===")
    print(f"Saved BM25 Index to: {PATHS['bm25_index']}")

build_sparse_bm25_index()

=== BUILDING SPARSE BM25 KEYWORD INDEX ===
Building BM25 Inverted Index...

=== SPARSE INDEX COMPLETE ===
Saved BM25 Index to: E:\Rasengan\College-Knowledge-Search\data\processed\bm25_index.pkl


In [4]:
faiss_exists = os.path.exists(PATHS["faiss_index"])
bm25_exists = os.path.exists(PATHS["bm25_index"])

print("=== STAGE 3 INDEX VERIFICATION ===")
print(f"FAISS Dense Vector Index: {' READY' if faiss_exists else ' MISSING'}")
print(f"BM25 Sparse Keyword Index: {' READY' if bm25_exists else ' MISSING'}")

if faiss_exists and bm25_exists:
    faiss_size = os.path.getsize(PATHS["faiss_index"]) / (1024 * 1024)
    bm25_size = os.path.getsize(PATHS["bm25_index"]) / (1024 * 1024)
    print(f"\nFAISS File Size: {faiss_size:.2f} MB")
    print(f"BM25 File Size: {bm25_size:.2f} MB")
    print("\n🚀 Stage 3 Complete! Ready for Stage 4: Multi-Stage Retrieval & Re-Ranking.")

=== STAGE 3 INDEX VERIFICATION ===
FAISS Dense Vector Index:  READY
BM25 Sparse Keyword Index:  READY

FAISS File Size: 3.45 MB
BM25 File Size: 7.34 MB

🚀 Stage 3 Complete! Ready for Stage 4: Multi-Stage Retrieval & Re-Ranking.
